# Lecture 2 Tutorial — Peru Fires, End-to-End with the Modern Geo Stack
## Session 2 · Interactive Maps + Modern Raster + Spatial SQL + Big Data

**Anzony Quispe · Jesus Gastañaduy · 2025**

---

### The story

We continue the Peru fires example from Lecture 1, but now we build a
single **end-to-end pipeline** that touches every library in Session 2:

1. **Load** the 2015 fires + Peru regions from Lecture 1 as **GeoParquet**
2. **Query** fires-per-region with **DuckDB spatial SQL**
3. **Scale up** to multi-year fires with **dask-geopandas**
4. **Add NDVI** context — a Landsat scene via **geowombat**
5. **Publish** the result in an interactive **folium** + **lonboard** map

By the end you will have one shareable HTML map that layers vector,
raster, and SQL-derived aggregates over Peru.

---

### Learning objectives

| # | You will be able to… |
|---|---------------------|
| 1 | Convert a shapefile / CSV to GeoParquet and see the size drop |
| 2 | Write a `ST_Within` spatial join in **SQL** using DuckDB |
| 3 | Run a spatial join in **parallel** with dask-geopandas |
| 4 | Compute **NDVI** on a Landsat scene with geowombat |
| 5 | Overlay all layers in one **folium** map with a layer switcher |
| 6 | Render a million-point layer with **lonboard** |

---
## 0 · Setup

All packages are declared in `pyproject.toml`. If any import fails,
follow `SETUP.md` (conda option is safest).

In [20]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

# Session 2 stack
import duckdb
import dask_geopandas as dgpd
import pyarrow
import folium

# Optional but recommended
try:
    from lonboard import Map, ScatterplotLayer
    HAS_LONBOARD = True
except ImportError:
    HAS_LONBOARD = False
    print('lonboard not installed — the last section will fall back to folium')

try:
    import geowombat as gw
    HAS_GEOWOMBAT = True
except ImportError:
    HAS_GEOWOMBAT = False
    print('geowombat not installed — the NDVI section will be conceptual')

print('geopandas    :', gpd.__version__)
print('duckdb       :', duckdb.__version__)
print('dask-geopandas:', dgpd.__version__)
print('folium       :', folium.__version__)

lonboard not installed — the last section will fall back to folium
geopandas    : 1.1.1
duckdb       : 1.5.4
dask-geopandas: 0.5.0
folium       : 0.20.0


---
## Step 1 · Load data and convert to GeoParquet

We use the **same 2015 Peru fires and administrative boundaries** from
Lecture 1. This time we save both to GeoParquet and compare file sizes.

In [21]:
YEAR = 2015

# Fires — VIIRS-SNPP CSV from NASA FIRMS
fires_csv_url = (
    f'https://firms.modaps.eosdis.nasa.gov/data/country/viirs-snpp/'
    f'{YEAR}/viirs-snpp_{YEAR}_Peru.csv'
)
fires_df = pd.read_csv(fires_csv_url)

fires = gpd.GeoDataFrame(
    fires_df,
    geometry=gpd.points_from_xy(fires_df.longitude, fires_df.latitude),
    crs='EPSG:4326'
)
print(f'Fire points : {len(fires):,}')

# Peru regions (Level 1)
peru = gpd.read_file(
    'https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_PER_1.json'
)
peru = peru[['GID_1', 'NAME_1', 'geometry']].copy()
print(f'Peru regions: {len(peru)}')

Fire points : 69,996
Peru regions: 26


In [22]:
os.makedirs('/tmp/geo_tutorial', exist_ok=True)

# --- Save both layers as GeoParquet ---
fires_path = '/tmp/geo_tutorial/fires_2015.parquet'
peru_path  = '/tmp/geo_tutorial/peru_regions.parquet'

fires.to_parquet(fires_path, compression='snappy')
peru.to_parquet(peru_path,   compression='snappy')

# --- Compare sizes with the equivalent shapefile ---
shp_path = '/tmp/geo_tutorial/fires_2015.shp'
fires.to_file(shp_path)

import glob
shp_bytes = sum(os.path.getsize(f) for f in glob.glob('/tmp/geo_tutorial/fires_2015.*') if not f.endswith('.parquet'))
pq_bytes  = os.path.getsize(fires_path)

print(f'Shapefile total: {shp_bytes/1e6:,.1f} MB')
print(f'GeoParquet     : {pq_bytes/1e6:,.1f} MB')
print(f'Ratio          : {shp_bytes/pq_bytes:.1f}x smaller')

Shapefile total: 46.1 MB
GeoParquet     : 3.0 MB
Ratio          : 15.3x smaller


---
## Step 2 · Spatial join in SQL with DuckDB

In Lecture 1 we did a spatial join with `gpd.sjoin`. Here we do the
**same operation in SQL** — no server, no PostGIS, no data copy.

DuckDB reads the GeoParquet files directly and executes the join with
its multi-threaded columnar engine.

In [24]:
con = duckdb.connect()
con.execute('INSTALL spatial;  LOAD spatial;')

# List loaded extensions
con.execute(
    "SELECT extension_name, loaded "
    "FROM duckdb_extensions() "
    "WHERE loaded = TRUE "
    "ORDER BY extension_name;"
).fetchdf()

,extension_name,loaded
0,core_functions,True
1,icu,True
2,json,True
3,parquet,True
4,spatial,True


In [26]:
sql = f"""
    -- Fires per Peru region, in SQL
    SELECT
        p.GID_1,
        p.NAME_1,
        COUNT(*)   AS n_fires,
        SUM(f.frp) AS total_frp,
        AVG(f.frp) AS mean_frp
    FROM   read_parquet('{fires_path}') AS f
    JOIN   read_parquet('{peru_path}')  AS p
      ON   ST_Within(f.geometry, p.geometry)
    GROUP  BY p.GID_1, p.NAME_1
    ORDER  BY n_fires DESC
    LIMIT  15;
"""
fires_by_region = con.execute(sql).fetchdf()

print('Top 15 Peru regions by 2015 fire detections:')
fires_by_region.round(1)

Top 15 Peru regions by 2015 fire detections:


,GID_1,NAME_1,n_fires,total_frp,mean_frp
0,PER.23_1,SanMartín,11557,102815.0,8.9
1,PER.26_1,Ucayali,11432,133180.8,11.6
2,PER.10_1,Huánuco,8716,111854.7,12.8
3,PER.17_1,Loreto,6522,56665.6,8.7
4,PER.12_1,Junín,6277,60266.0,9.6
5,PER.18_1,MadredeDios,3730,50011.8,13.4
6,PER.8_1,Cusco,3513,36665.6,10.4
7,PER.20_1,Pasco,2869,33388.9,11.6
8,PER.6_1,Cajamarca,2301,20289.6,8.8
9,PER.13_1,LaLibertad,1949,13872.8,7.1


> **Why this matters:** the same query would work on a 100 GB dataset
> stored on S3, using only the memory required for the result. DuckDB
> only scans the columns and rows it needs (predicate pushdown).

---
## Step 3 · Scale up with dask-geopandas

To simulate multi-year fire data, we synthetically triple the 2015
dataset. In a real project you would concatenate 2012–2024 CSVs from
NASA FIRMS — that would be ~250 000 points, still fits in RAM but the
`sjoin` starts to slow down.

We then partition the data across CPU cores and run the same join
in **parallel**.

In [27]:
# Concatenate the same year 3 times → 3x the size for demonstration
fires_big = pd.concat([fires, fires, fires], ignore_index=True)
fires_big = gpd.GeoDataFrame(fires_big, crs='EPSG:4326')
print(f'Simulated multi-year fires: {len(fires_big):,} points')

# --- Convert to a dask-geopandas partitioned frame ---
fires_dgpd = dgpd.from_geopandas(fires_big, npartitions=4)
peru_dgpd  = dgpd.from_geopandas(peru,      npartitions=1)

print(f'Partitions : {fires_dgpd.npartitions}')
print(f'Type       : {type(fires_dgpd).__name__}')

Simulated multi-year fires: 209,988 points
Partitions : 4
Type       : GeoDataFrame


In [28]:
# Spatial join, executed lazily
joined = fires_dgpd.sjoin(peru_dgpd, predicate='within')

# Group + count — still lazy
counts_lazy = joined.groupby('NAME_1').size()

# Trigger execution across CPU cores
counts = counts_lazy.compute().sort_values(ascending=False)

print('Fires per region (parallel, 3x simulated data):')
counts.head(10)

Fires per region (parallel, 3x simulated data):


NAME_1
SanMartín      34671
Ucayali        34296
Huánuco        26148
Loreto         19566
Junín          18831
MadredeDios    11190
Cusco          10539
Pasco           8607
Cajamarca       6903
LaLibertad      5847
dtype: int64

---
## Step 4 · NDVI context with `geowombat`

**NDVI** (Normalised Difference Vegetation Index) tells us how green
the land is — high values (~0.7) mean dense forest, low values
(~0.1) mean bare soil or urban.

We compute NDVI for a Landsat scene over the Peruvian Amazon, then
overlay it with the fire aggregates from Step 2.

> **Note.** The full Landsat scene is ~2 GB. This section is
> **conceptual** in class — it shows the code, but does not download
> the scene. In a homework you would swap in a real URL.

In [11]:
if HAS_GEOWOMBAT:
    # Placeholder URLs — replace with a real Landsat 8 tile over Peru
    # e.g. from https://landsatlook.usgs.gov/
    landsat_red = 'https://example.com/LC08_L2SP_006067_20150801_B4.TIF'
    landsat_nir = 'https://example.com/LC08_L2SP_006067_20150801_B5.TIF'

    # This is the pattern — commented so we don't hit the network in class
    # with gw.open([landsat_red, landsat_nir],
    #              band_names=['red', 'nir'],
    #              chunks=1024) as src:
    #     ndvi = src.gw.norm_diff('nir', 'red')
    #     ndvi.gw.save('/tmp/geo_tutorial/ndvi.tif',
    #                  overwrite=True, num_workers=4)
    #
    # print('NDVI written to /tmp/geo_tutorial/ndvi.tif')

    print('geowombat is installed — swap in a real Landsat URL to run.')
else:
    print('geowombat not installed — see SETUP.md to add it.')

geowombat is installed — swap in a real Landsat URL to run.


### What geowombat did for us

- **Streamed** only the requested area (COG range requests)
- Computed NDVI in **chunks** using dask
- Wrote a **cloud-optimised** GeoTIFF as output
- All in **3 lines** of code

The equivalent in raw rasterio would be ~40 lines of manual band-by-band
reading, cropping, dividing, and writing.

---
## Step 5 · Publish an interactive folium map

We join the SQL results back to the Peru regions and render a
**choropleth** of fires per region with a hover tooltip.

In [29]:
# Merge the SQL result onto the Peru polygons
peru_with_counts = peru.merge(fires_by_region, on=['GID_1', 'NAME_1'], how='left')
peru_with_counts['n_fires']   = peru_with_counts['n_fires'].fillna(0).astype(int)
peru_with_counts['total_frp'] = peru_with_counts['total_frp'].fillna(0)

peru_with_counts[['NAME_1','n_fires','total_frp']].sort_values('n_fires', ascending=False).head(6)

,NAME_1,n_fires,total_frp
22,SanMartín,11557,102814.96
25,Ucayali,11432,133180.77
9,Huánuco,8716,111854.69
16,Loreto,6522,56665.60
11,Junín,6277,60265.95
17,MadredeDios,3730,50011.79


In [30]:
m = folium.Map(location=[-9.19, -75.02], zoom_start=5, tiles='CartoDB positron')

# Choropleth of fires
folium.Choropleth(
    geo_data     = peru_with_counts.__geo_interface__,
    data         = peru_with_counts,
    columns      = ['GID_1', 'n_fires'],
    key_on       = 'feature.properties.GID_1',
    fill_color   = 'YlOrRd',
    fill_opacity = 0.75,
    line_opacity = 0.4,
    legend_name  = f'VIIRS active fires — Peru {YEAR}',
    name         = 'Fires per region',
).add_to(m)

# Hover popups
folium.GeoJson(
    peru_with_counts,
    name='Hover info',
    style_function=lambda x: {'fillOpacity': 0, 'color': 'transparent'},
    tooltip=folium.GeoJsonTooltip(
        fields=['NAME_1', 'n_fires', 'total_frp'],
        aliases=['Region', 'Fires', 'Total FRP (MW)'],
        localize=True,
    )
).add_to(m)

# Sample of fire points for context (take 500 for speed)
sample = fires.sample(min(500, len(fires)), random_state=1)
fg = folium.FeatureGroup(name='Sample fires (500)')
for _, row in sample.iterrows():
    folium.CircleMarker(
        location=[row.latitude, row.longitude],
        radius=1.5, color='#dc2626', fill=True, fill_opacity=0.6
    ).add_to(fg)
fg.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

# Save + display
m.save('/Users/anzony.quisperojas/Documents/GitHub/python/GeoAgent/Lectures/peru_fires_2015.html')
m

---
## Step 6 · Render every fire point with lonboard

Folium starts to feel sluggish beyond ~10 000 points. For the full
70 000 fires (or 250 000 across multiple years), we use **lonboard**
which pushes the data as GeoArrow and renders on the GPU.

In [19]:
if HAS_LONBOARD:
    # Reproject to Web Mercator is optional — lonboard handles 4326
    layer = ScatterplotLayer.from_geopandas(
        fires,
        get_fill_color=[220, 40, 40, 180],
        get_radius=800,
        radius_units='meters',
    )

    map_lb = Map(
        layer,
        view_state={'longitude': -75.0, 'latitude': -9.0, 'zoom': 5},
    )
    display(map_lb)
else:
    # Fallback: show the same points on a folium heatmap
    from folium.plugins import HeatMap
    m2 = folium.Map(location=[-9.19, -75.02], zoom_start=5, tiles='CartoDB dark_matter')
    HeatMap(fires[['latitude','longitude']].values, radius=6, blur=8).add_to(m2)
    display(m2)
    m2.save('/Users/anzony.quisperojas/Documents/GitHub/python/GeoAgent/Lectures/peru_fires_2015_all.html')


---
## Part 7 · Exercises

Extend the pipeline you built above. Each exercise touches one layer
of the stack.

### Exercise 1 — Try multiple years

NASA FIRMS has data from 2012 onwards. Download 2015 + 2020 + 2023,
save each as a **partitioned GeoParquet** file (`fires_YYYY.parquet`),
and use DuckDB to compute fires per region **per year** with a `GROUP BY
year, NAME_1` clause.

Hint:

```python
years = [2015,2025]
for y in years:
    df = pd.read_csv(f'https://firms.modaps.eosdis.nasa.gov/...')
    df['year'] = y
    gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude),
                     crs='EPSG:4326').to_parquet(f'/tmp/fires_{y}.parquet')
```

Then DuckDB can read all files with a **glob**:

```sql
SELECT year, NAME_1, COUNT(*) AS n
FROM   read_parquet('/tmp/fires_*.parquet') f
JOIN   read_parquet('/tmp/peru_regions.parquet') p
  ON   ST_Within(f.geometry, p.geometry)
GROUP  BY year, NAME_1
ORDER  BY year, n DESC;
```

In [15]:
# Your code here


### Exercise 2 — SQL vs geopandas vs dask

Time the three implementations of the same spatial join on 3x fires:

1. `gpd.sjoin(fires_big, peru, predicate='within')`
2. DuckDB `ST_Within` query
3. `dgpd.from_geopandas(...).sjoin(...).compute()`

Report the runtime in seconds. Which one wins for this dataset size?

In [16]:
import time

# 1) plain geopandas
t0 = time.time()
_ = gpd.sjoin(fires_big, peru, predicate='within')
t_gpd = time.time() - t0

# 2) DuckDB — your code here
t_duck = None

# 3) dask-geopandas — your code here
t_dask = None

print(f'geopandas    : {t_gpd:.2f} s')
print(f'duckdb       : {t_duck}')
print(f'dask-geopandas: {t_dask}')

geopandas    : 0.10 s
duckdb       : None
dask-geopandas: None


### Exercise 3 — Replicate a paper

Pick **one** paper from Section 5 of `Lecture2.ipynb` and outline how
you would replicate its main figure in a Jupyter notebook using the
tools in this course. Include:

1. Which open dataset(s) you would fetch
2. Which library maps to which paper's step
3. What the simplest version of the result would look like (one country,
   one year, one panel of the figure)

Examples to start from:

- **Storeygard (2016)** — download DMSP nightlights + African road
  shapefile → `rasterstats.zonal_stats` → panel regression
- **Assunção et al. (2020)** — download INPE PRODES rasters + IBGE
  municipality shapes → zonal stats per year → DiD around 2008 policy
- **Weiss et al. (2020)** — download Malaria Atlas friction raster →
  `skimage.graph.route_through_array` from every populated place

*Write your outline here (as a markdown cell).*